In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import polars as pl
from influxdb_client  import InfluxDBClient , WriteOptions
import yaml
import requests


print("CWD:", Path.cwd())

from dotenv import load_dotenv
load_dotenv("mlops_services/.env.mlops")

CWD: /home/legacy/Projects/Weather_forecasting_with_MLOps


True

In [3]:
from mlops_services.src.utils import configs 
config = configs.Config.from_yaml(
            "mlops_services/config/model_params.yaml"
        )


In [ ]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
from mlops_services.src.utils import  logs
logger = logs.setup_logger("test")
logger.info("Hello")

logger2 = logs.setup_logger("test2")
logger2.info("Hellooo")


In [ ]:
%run mlops_services/src/pipelines/train/a_retrieve.py local BLR "2023-01-01 00:00" "2025-06-30 23:00"

In [ ]:
%run mlops_services/src/pipelines/train/b_split.py

In [ ]:
%run mlops_services/src/pipelines/train/c_preprocess.py

In [ ]:
%tb

In [ ]:
df_train = pl.read_parquet('mlops_services/data/final/train.parquet')
df_train

In [ ]:
%run mlops_services/src/pipelines/train/d_train.py XGB

In [4]:
from mlops_services.src.components.model_training import Trainer
from mlops_services.src.utils import  logs
logger = logs.setup_logger("trainer")
model_trainer = Trainer(config, logger, "train")

Support for PyTorch based likelihood models not available. To enable them, install "darts", "u8darts[torch]" or "u8darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
Support for Torch based models not available. To enable them, install "darts", "u8darts[torch]" or "u8darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
/home/legacy/Projects/Weather_forecasting_with_MLOps/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2025-10-11 18:53:52,155 - INFO - Using past features: ['cloud_cover', 'dew_point_2m', 'et0_fao_evapotranspiration', 'generationtime_ms', 'rain', 'relative_humidity_2m', 'surface_pressure', 'vapour_pressure_deficit', 'weather_code', 'wind_dire

In [5]:
model_trainer.train('XGB')

2025-10-11 18:54:42,899 - INFO - Algorithm selected: XGB
2025-10-11 18:54:42,902 - INFO - Training model for temperature_2m
2025-10-11 18:59:01,910 - INFO - Training Successfully completed in: 265.0392 seconds


In [6]:
model_trainer.log_experiment()

2025-10-11 18:59:06,203 - INFO - Tracking experiment using MLflow


type :  <class 'list'>


2025/10/11 18:59:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'Forecaster_XGB' already exists. Creating a new version of this model...
2025/10/11 18:59:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Forecaster_XGB, version 5
Created version '5' of model 'Forecaster_XGB'.
2025-10-11 18:59:19,291 - INFO - Experiment logged with run_id: 2b34b2bc95494e7ba12059f1c951570e
INFO:trainer:Experiment logged with run_id: 2b34b2bc95494e7ba12059f1c951570e


🏃 View run XGB_2025_10_11_18_59_07 at: https://dagshub.com/akasharan-a/Weather_forecasting_with_MLOps.mlflow/#/experiments/2/runs/2b34b2bc95494e7ba12059f1c951570e
🧪 View experiment at: https://dagshub.com/akasharan-a/Weather_forecasting_with_MLOps.mlflow/#/experiments/2


In [8]:
model_trainer.save()

2025-10-11 18:59:35,662 - INFO - Saving Model....
INFO:trainer:Saving Model....
2025/10/11 18:59:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025-10-11 18:59:35,880 - INFO - Model saved successfully
INFO:trainer:Model saved successfully


In [39]:
from mlops_services.src.components.model_evaluation import Evaluator
from mlops_services.src.utils import  logs
logger = logs.setup_logger("tester")
model_evaluate = Evaluator(config, logger, "train")

In [40]:
model_evaluate.load_model(source='online')

2025-10-11 19:52:34,467 - INFO - Loading Forescater from online registry..
INFO:tester:Loading Forescater from online registry..
2025-10-11 19:52:40,749 - INFO - Successful!!
INFO:tester:Successful!!


In [50]:
model_evaluate.evaluate_model()

2025-10-11 20:03:00,930 - INFO - Backtesting results : {'mape': 0.7906908414336469, 'mae': 0.18460779733886568, 'rmse': 0.18460779733886568}
INFO:tester:Backtesting results : {'mape': 0.7906908414336469, 'mae': 0.18460779733886568, 'rmse': 0.18460779733886568}


In [48]:
mape.__name__

'mape'

In [45]:
a = ['a','b','c'] 
b =[1,2,3]
dict(zip(a,b))

{'a': 1, 'b': 2, 'c': 3}

In [ ]:
type(model_evaluate.forecaster_packaged)

In [ ]:
model_evaluate.forecaster_packaged._model_meta._signature

In [ ]:
model_evaluate.forecaster.transform_input

In [ ]:
pl.from_dicts(model_evaluate.df.to_dicts())

In [35]:
hist_y,hist_X_past,hist_X_future = model_evaluate.forecaster.transform_input(model_evaluate.df)
y_hat = model_evaluate.forecaster.model.backtest(series=hist_y,past_covariates=hist_X_past,future_covariates=hist_X_future,
                                        forecast_horizon=1,stride=1,last_points_only =True,retrain=False)

In [36]:
y_hat

np.float64(0.7906908414336469)

In [ ]:
model_evaluate.forecaster

time,temperature_2m
datetime[ns],f32
2024-11-14 18:00:00,19.980089
2024-11-14 19:00:00,19.398741
2024-11-14 20:00:00,19.554173
2024-11-14 21:00:00,19.726078
2024-11-14 22:00:00,19.372982
…,…
2024-11-15 13:00:00,22.183125
2024-11-15 14:00:00,21.031219
2024-11-15 15:00:00,20.632139


In [ ]:
import mlflow
mlflow.end_run()

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Fetch all registered models
registered_models = client.search_registered_models()
registered_models


In [ ]:
for model in registered_models:
    model_name = model.name
    # Fetch all versions of this model
    versions = client.search_model_versions(filter_string=f"name='{model_name}'")
    
    # Optionally transition versions to 'Archived' stage if needed
    for v in versions:
        if v.current_stage in ("Staging", "Production"):
            client.transition_model_version_stage(
                name=model_name,
                version=v.version,
                stage="Archived",
                archive_existing_versions=False
            )
    
    # Delete all model versions
    for v in versions:
        client.delete_model_version(name=model_name, version=v.version)
        
    # Delete the registered model
    client.delete_registered_model(name=model_name)
    print(f"Deleted model and all versions: {model_name}")
